# Modelo Random Forest – Tempo de Estadia (TE) da soja exportada

Lê o `dataset_modelo.parquet` gerado pelo `CreateDataset_v2.ipynb` (1 linha = 1 atracação, 2019–2025).

**Target:** `lead_time_h` = TE = T1 + T2 + T3 + T4 (chegada à área de fundeio → desatracação), em horas.
O modelo é treinado em `log1p(TE)` por causa da cauda longa; as métricas são reportadas em horas.

**Cenário de uso:** prever o TE no momento da chegada do navio. Por isso só entram features conhecidas nesse
momento (ver `CreateDataset_v2.ipynb`).

**Validação temporal:** treino = 2019–2024, teste = 2025. Nunca split aleatório: navios que chegam na mesma
semana dividem a mesma fila, e o split aleatório deixaria o modelo "ver" o futuro.

Etapas:
1. Separação treino/teste
2. Baselines (mediana global e mediana por complexo × mês)
3. Ajuste de hiperparâmetros com validação temporal **dentro do treino**
4. Avaliação no teste (usado uma única vez para o modelo principal)
5. Incerteza da comparação com o baseline (bootstrap por semanas)
6. Importância das features (por bloco e individual)
7. Efeito parcial do mês de chegada
8. Análise de sensibilidade (retreino sem grupos de features, mesmos hiperparâmetros)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

FIG = Path.cwd() / "figuras"
FIG.mkdir(exist_ok=True)
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman"], "font.size": 10})

df = pd.read_parquet("dataset_modelo.parquet").sort_values("dt_chegada").reset_index(drop=True)
print(f"{len(df)} atracações | {df['dt_chegada'].min():%d/%m/%Y} a {df['dt_chegada'].max():%d/%m/%Y}")
df.groupby("ano").size().rename("atracações").to_frame().T

### Features

Ficam de fora: identificadores, T1–T4 (somados são o próprio target) e as colunas de alta cardinalidade
(`Porto Atracação`, `SGUF`), já representadas por `Complexo Portuário` e `Região Geográfica`.

In [ ]:
BLOCOS = {
    "Sazonalidade": ["mes", "trimestre", "semana_ano", "dia_semana", "sin_doy", "cos_doy", "periodo_safra"],
    "Porto": ["Complexo Portuário", "Tipo da Autoridade Portuária", "Região Geográfica"],
    "Carga": ["peso_soja_t", "peso_total_embarcado_t", "pct_soja", "n_lotes", "n_mercadorias", "n_destinos",
              "flag_destino_hub", "continente_destino"],
    "Congestionamento": ["fila_na_chegada", "chegadas_7d", "lead_time_medio_30d", "horas_chuva_7d"],
    "Navio": ["atracacoes_imo_365d"],
}
CATEGORICAS = ["periodo_safra", "Complexo Portuário", "Tipo da Autoridade Portuária", "Região Geográfica",
               "continente_destino"]
FEATURES = [col for cols in BLOCOS.values() for col in cols]


def montar_x(dados, features, colunas=None):
    ## One-hot das categóricas; reindex garante as mesmas colunas em qualquer recorte
    x = pd.get_dummies(dados[features], columns=[c for c in CATEGORICAS if c in features], dtype=int)
    return x if colunas is None else x.reindex(columns=colunas, fill_value=0)


X = montar_x(df, FEATURES)
COLUNAS_X = X.columns
y = np.log1p(df["lead_time_h"])
print(f"{len(FEATURES)} features -> {X.shape[1]} colunas após one-hot")

### 1. Separação treino / teste

In [ ]:
ANO_TESTE = df["ano"].max()
treino = df["ano"] < ANO_TESTE
teste = ~treino
print(f"Treino: {treino.sum()} atracações ({df.loc[treino, 'dt_chegada'].min():%m/%Y} a "
      f"{df.loc[treino, 'dt_chegada'].max():%m/%Y})")
print(f"Teste : {teste.sum()} atracações ({df.loc[teste, 'dt_chegada'].min():%m/%Y} a "
      f"{df.loc[teste, 'dt_chegada'].max():%m/%Y})")

### 2. Baselines

- **Mediana global**: o mesmo valor para todos os navios (a referência mais ingênua).
- **Mediana por complexo × mês**: o que um analista faria sem modelo, usando o histórico daquele complexo
  naquele mês. O Random Forest só agrega valor se superar essa referência.

In [ ]:
real = df.loc[teste, "lead_time_h"]
baseline_global = pd.Series(df.loc[treino, "lead_time_h"].median(), index=real.index)

mediana_complexo_mes = df[treino].groupby(["Complexo Portuário", "mes"])["lead_time_h"].median()
mediana_complexo = df[treino].groupby("Complexo Portuário")["lead_time_h"].median()
baseline = pd.Series(
    [mediana_complexo_mes.get((c, m), np.nan) for c, m in zip(df.loc[teste, "Complexo Portuário"], df.loc[teste, "mes"])],
    index=real.index)
baseline = baseline.fillna(df.loc[teste, "Complexo Portuário"].map(mediana_complexo)).fillna(baseline_global)

### 3. Ajuste de hiperparâmetros

`TimeSeriesSplit` dentro do treino: cada fold treina no passado e valida no período seguinte.
O conjunto de teste (2025) não participa desta etapa.

In [ ]:
busca = GridSearchCV(
    RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42),
    param_grid={"max_features": [0.3, 0.6, 1.0], "min_samples_leaf": [1, 5, 10]},
    cv=TimeSeriesSplit(n_splits=4),
    scoring="neg_mean_absolute_error",
)
busca.fit(X[treino], y[treino])

print("Melhores parâmetros:", busca.best_params_)
Df_Busca = (pd.DataFrame(busca.cv_results_)[["params", "mean_test_score", "std_test_score"]]
            .sort_values("mean_test_score", ascending=False))
Df_Busca["params"] = Df_Busca["params"].astype(str)
Df_Busca.round(4)

### 4. Avaliação no teste

MAE = erro médio em horas; RMSE penaliza mais os erros grandes; MdAE = erro do caso típico (mediana);
R² = fração da variância explicada, em horas; R² (log) = o mesmo na escala em que o modelo foi treinado.

In [ ]:
rf = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42, **busca.best_params_)
rf.fit(X[treino], y[treino])
previsto = pd.Series(np.expm1(rf.predict(X[teste])), index=real.index)


def metricas(real, previsto):
    return {"MAE (h)": mean_absolute_error(real, previsto),
            "RMSE (h)": root_mean_squared_error(real, previsto),
            "MdAE (h)": np.median(np.abs(real - previsto)),
            "R²": r2_score(real, previsto),
            "R² (log)": r2_score(np.log1p(real), np.log1p(previsto))}


Df_Metricas = pd.DataFrame({"Baseline 1 – mediana global": metricas(real, baseline_global),
                            "Baseline 2 – mediana complexo × mês": metricas(real, baseline),
                            "Random Forest": metricas(real, previsto)}).T
Df_Metricas.round(3)

In [ ]:
## Erro por mês de chegada no teste
Df_Teste = df.loc[teste, ["IDAtracacao", "dt_chegada", "mes", "Complexo Portuário", "lead_time_h"]].assign(
    previsto_rf=previsto, previsto_baseline=baseline)
Df_Erro_Mes = Df_Teste.groupby("mes").apply(lambda g: pd.Series({
    "n": len(g),
    "real_mediano_h": g["lead_time_h"].median(),
    "previsto_mediano_h": g["previsto_rf"].median(),
    "MAE_rf_h": mean_absolute_error(g["lead_time_h"], g["previsto_rf"]),
    "MAE_baseline_h": mean_absolute_error(g["lead_time_h"], g["previsto_baseline"]),
}), include_groups=False)
Df_Erro_Mes.round(1)

### 5. Incerteza da comparação com o baseline

Bootstrap por **semanas de chegada** (2.000 reamostragens): navios da mesma semana compartilham a fila e não são
independentes, então a reamostragem é feita em blocos semanais. Diferença negativa = Random Forest erra menos.

In [ ]:
semanas = df.loc[teste, "dt_chegada"].dt.to_period("W").astype(str).to_numpy()
erro_rf = np.abs(real.to_numpy() - previsto.to_numpy())
erro_bl = np.abs(real.to_numpy() - baseline.to_numpy())
indices_semana = {s: np.flatnonzero(semanas == s) for s in np.unique(semanas)}
lista_semanas = list(indices_semana)

rng = np.random.default_rng(42)
dif_mae, dif_mdae = [], []
for _ in range(2000):
    idx = np.concatenate([indices_semana[s] for s in rng.choice(lista_semanas, size=len(lista_semanas))])
    dif_mae.append(erro_rf[idx].mean() - erro_bl[idx].mean())
    dif_mdae.append(np.median(erro_rf[idx]) - np.median(erro_bl[idx]))

Df_Bootstrap = pd.DataFrame({
    "diferenca_observada_h": [erro_rf.mean() - erro_bl.mean(), np.median(erro_rf) - np.median(erro_bl)],
    "ic95_inferior_h": [np.percentile(dif_mae, 2.5), np.percentile(dif_mdae, 2.5)],
    "ic95_superior_h": [np.percentile(dif_mae, 97.5), np.percentile(dif_mdae, 97.5)],
    "pct_reamostras_rf_melhor": [100 * np.mean(np.array(dif_mae) < 0), 100 * np.mean(np.array(dif_mdae) < 0)],
}, index=["MAE (RF − baseline 2)", "MdAE (RF − baseline 2)"])
print(f"Semanas no teste: {len(lista_semanas)}")
Df_Bootstrap.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].scatter(real, previsto, s=6, alpha=0.35, color="0.3")
limite = np.percentile(real, 99)
ax[0].plot([0, limite], [0, limite], "k--", lw=1)
ax[0].set(xlim=(0, limite), ylim=(0, limite), xlabel="TE real (horas)", ylabel="TE previsto (horas)",
          title=f"(a) Real × previsto – teste {ANO_TESTE}")
(Df_Teste.groupby("mes")[["lead_time_h", "previsto_rf", "previsto_baseline"]].median()
 .rename(columns={"lead_time_h": "Real", "previsto_rf": "Random Forest", "previsto_baseline": "Baseline 2"})
 .rename(index=lambda m: MESES[m - 1])
 .plot(ax=ax[1], style=["-o", "--s", ":^"], color=["black", "0.35", "0.6"], ms=4))
meses_teste = sorted(Df_Teste["mes"].unique())
ax[1].set_xticks(range(len(meses_teste)), [MESES[m - 1] for m in meses_teste])
ax[1].set(xlabel="mês de chegada", ylabel="TE mediano (horas)", title="(b) Mediana mensal no teste")
ax[1].legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(FIG / "fig_real_previsto.png", dpi=200)
plt.show()

### 6. Importância das features

Importância por permutação no teste: quanto o erro piora quando embaralhamos uma feature (ou um bloco inteiro).
É mais confiável que o `feature_importances_` nativo do RF, que favorece variáveis contínuas.
**Importância preditiva não é causalidade.**

In [ ]:
rng = np.random.default_rng(42)
X_teste, y_teste = X[teste], y[teste]
mae_base = mean_absolute_error(y_teste, rf.predict(X_teste))


def colunas_do_bloco(cols):
    return [c for c in COLUNAS_X if c in cols or any(c.startswith(f"{cat}_") for cat in cols)]


importancia_bloco = {}
for bloco, cols in BLOCOS.items():
    colunas_x = colunas_do_bloco(cols)
    aumentos = []
    for _ in range(10):
        X_perm = X_teste.copy()
        X_perm[colunas_x] = X_perm[colunas_x].values[rng.permutation(len(X_perm))]
        aumentos.append(mean_absolute_error(y_teste, rf.predict(X_perm)) - mae_base)
    importancia_bloco[bloco] = np.mean(aumentos)

Df_Importancia_Bloco = (pd.Series(importancia_bloco, name="aumento_MAE_log").sort_values(ascending=False)
                        .to_frame().assign(participacao=lambda d: d["aumento_MAE_log"] / d["aumento_MAE_log"].sum()))
Df_Importancia_Bloco.round(4)

In [ ]:
perm = permutation_importance(rf, X_teste, y_teste, scoring="neg_mean_absolute_error",
                              n_repeats=10, random_state=42, n_jobs=-1)
Df_Importancia = (pd.DataFrame({"feature": COLUNAS_X, "importancia": perm.importances_mean,
                                "desvio": perm.importances_std})
                  .sort_values("importancia", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
Df_Importancia_Bloco["participacao"].mul(100)[::-1].plot.barh(ax=ax[0], color="0.45")
ax[0].set(xlabel="% da importância total", title="(a) Por bloco de features")
Df_Importancia.head(12).set_index("feature")["importancia"][::-1].plot.barh(ax=ax[1], color="0.45")
ax[1].set(xlabel="aumento do MAE (escala log)", ylabel="", title="(b) 12 features mais importantes")
ax[1].tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.savefig(FIG / "fig_importancia.png", dpi=200)
plt.show()
Df_Importancia.head(15).round(4)

### 7. Efeito parcial do mês de chegada

Pergunta do TCC: **quanto o mês de chegada muda o TE previsto?**

Para cada mês m, todas as atracações do teste recebem as features de calendário de uma chegada no dia 15 do
mês m, mantendo porto, carga e congestionamento como estavam (dependência parcial). A média das previsões
mostra o **efeito direto do calendário** no modelo. A sazonalidade também age **indiretamente**, pela fila,
que aparece no bloco "Congestionamento". Isso descreve o comportamento do modelo, não uma relação causal.

In [ ]:
def calendario(dados, datas):
    dados = dados.copy()
    dados["mes"] = datas.dt.month
    dados["trimestre"] = datas.dt.quarter
    dados["semana_ano"] = datas.dt.isocalendar().week.astype(int)
    dados["sin_doy"] = np.sin(2 * np.pi * datas.dt.dayofyear / 365.25)
    dados["cos_doy"] = np.cos(2 * np.pi * datas.dt.dayofyear / 365.25)
    dados["periodo_safra"] = np.select([dados["mes"].between(2, 6), dados["mes"].between(7, 10)],
                                       ["pico_soja", "safrinha_milho"], default="entressafra")
    return dados


base_teste = df[teste]
efeito = []
for mes in range(1, 13):
    datas = pd.to_datetime(dict(year=base_teste["ano"], month=mes, day=15))
    previsto_mes = np.expm1(rf.predict(montar_x(calendario(base_teste, datas), FEATURES, COLUNAS_X)))
    efeito.append({"mes": mes, "te_previsto_medio_h": previsto_mes.mean()})

Df_Efeito_Sazonal = pd.DataFrame(efeito).set_index("mes")
media_anual = Df_Efeito_Sazonal["te_previsto_medio_h"].mean()
Df_Efeito_Sazonal["diferenca_vs_media_h"] = Df_Efeito_Sazonal["te_previsto_medio_h"] - media_anual
Df_Efeito_Sazonal["diferenca_vs_media_%"] = 100 * Df_Efeito_Sazonal["diferenca_vs_media_h"] / media_anual
Df_Efeito_Sazonal["te_real_mediano_todos_anos_h"] = df.groupby("mes")["lead_time_h"].median()
Df_Efeito_Sazonal.index = MESES

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(MESES, Df_Efeito_Sazonal["diferenca_vs_media_%"],
       color=["0.3" if v > 0 else "0.7" for v in Df_Efeito_Sazonal["diferenca_vs_media_%"]])
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="mês de chegada", ylabel="% em relação à média anual")
plt.tight_layout()
plt.savefig(FIG / "fig_efeito_mes.png", dpi=200)
plt.show()
Df_Efeito_Sazonal.round(1)

### 8. Análise de sensibilidade

O modelo principal foi definido antes desta etapa e **não é escolhido** a partir dela. Aqui cada variante é
retreinada com os mesmos hiperparâmetros, retirando um grupo de features, para responder:
- **Sem calendário:** quanto o modelo perde sem as features de sazonalidade?
- **Sem congestionamento:** quanto perde sem a situação do porto na chegada?
- **Carga estrita:** as features de carga vêm de registros feitos após o embarque (aproximam o plano de carga
  conhecido no line-up). Esta variante mantém só `peso_soja_t`, a mais próxima da quantidade programada.
- **Somente calendário e porto:** o que se consegue só com informação de calendário e localização.

In [ ]:
VARIANTES = {
    "Modelo principal (todas as features)": FEATURES,
    "Sem calendário": [f for f in FEATURES if f not in BLOCOS["Sazonalidade"]],
    "Sem congestionamento": [f for f in FEATURES if f not in BLOCOS["Congestionamento"]],
    "Carga estrita (só peso de soja)": [f for f in FEATURES if f not in BLOCOS["Carga"] or f == "peso_soja_t"],
    "Somente calendário e porto": BLOCOS["Sazonalidade"] + BLOCOS["Porto"],
}
sensibilidade = {}
for nome, features in VARIANTES.items():
    x_var = montar_x(df, features)
    modelo = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42, **busca.best_params_)
    modelo.fit(x_var[treino], y[treino])
    sensibilidade[nome] = {"features": len(features),
                           **metricas(real, pd.Series(np.expm1(modelo.predict(x_var[teste])), index=real.index))}
Df_Sensibilidade = pd.DataFrame(sensibilidade).T
Df_Sensibilidade.round(3)

### Salvando resultados

In [ ]:
with pd.ExcelWriter("Resultados_RandomForest.xlsx", engine="openpyxl") as writer:
    Df_Metricas.to_excel(writer, sheet_name="Metricas")
    Df_Bootstrap.to_excel(writer, sheet_name="Bootstrap_Baseline")
    Df_Sensibilidade.to_excel(writer, sheet_name="Sensibilidade")
    Df_Importancia_Bloco.to_excel(writer, sheet_name="Importancia_Bloco")
    Df_Importancia.to_excel(writer, sheet_name="Importancia_Feature", index=False)
    Df_Efeito_Sazonal.to_excel(writer, sheet_name="Efeito_Sazonal")
    Df_Erro_Mes.to_excel(writer, sheet_name="Erro_por_Mes")
    Df_Busca.to_excel(writer, sheet_name="Busca_Hiperparametros", index=False)
    Df_Teste.to_excel(writer, sheet_name="Previsoes_Teste", index=False)